# Healthcare Commercial Analytics: Sales Force Effectiveness & HCP Targeting
## Exploratory Data Analysis

**Objective:** This notebook explores physician prescribing behavior to understand data distributions, specialty and regional prescribing patterns, and the relationship between prescribing volume, cost, and sales rep engagement. Insights from this EDA inform downstream HCP (Health Care Provider) segmentation and sales force targeting strategy.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("default")

## 1. Load Dataset

In [ ]:
df = pd.read_csv("../data/processed/analytics_data.csv")

print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

In [ ]:
df.dtypes

In [ ]:
df.describe(include="number").T

## 2. Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_summary[missing_summary["missing_count"] > 0].sort_values("missing_count", ascending=False)

**Observation:** Missing values are minimal or absent, indicating that the upstream preprocessing pipeline successfully handled data cleaning (e.g., imputation, filtering of incomplete records). The dataset is ready for analysis without further cleaning.

## 3. Distribution Analysis

Examining the distributions of key prescribing metrics to understand the underlying shape of physician-level activity.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df["Tot_Clms"], bins=50, color="steelblue", ax=ax)
ax.set_title("Distribution of Total Claims per Physician")
ax.set_xlabel("Total Claims")
ax.set_ylabel("Number of Physicians")
plt.tight_layout()
plt.show()

**Business observation:** The distribution is right-skewed — most physicians prescribe a relatively small number of claims, while a small subset drives disproportionately high claim volume. This long tail signals an opportunity to prioritize high-volume prescribers in sales targeting.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df["Tot_Drug_Cst"], bins=50, color="darkorange", ax=ax)
ax.set_title("Distribution of Total Drug Cost per Physician")
ax.set_xlabel("Total Drug Cost ($)")
ax.set_ylabel("Number of Physicians")
plt.tight_layout()
plt.show()

**Business observation:** Total drug cost is heavily right-skewed as well, consistent with claims volume. A concentrated group of physicians represents a disproportionate share of total drug spend, making them high-value targets for commercial engagement.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df["Tot_Benes"], bins=50, color="seagreen", ax=ax)
ax.set_title("Distribution of Total Beneficiaries per Physician")
ax.set_xlabel("Total Beneficiaries")
ax.set_ylabel("Number of Physicians")
plt.tight_layout()
plt.show()

**Business observation:** The number of beneficiaries served per physician follows a similar skewed pattern, suggesting that patient reach and prescribing volume are closely linked — physicians with larger patient panels tend to generate more claims.

## 4. Specialty Analysis

Identifying which physician specialties drive the highest prescribing volume.

In [ ]:
top_specialties = (
    df.groupby("Prscrbr_Type")["Tot_Clms"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=top_specialties.values, y=top_specialties.index, color="steelblue", ax=ax)
ax.set_title("Top 10 Physician Specialties by Total Claims")
ax.set_xlabel("Total Claims")
ax.set_ylabel("Specialty")
plt.tight_layout()
plt.show()

**Business interpretation:** Prescribing volume is concentrated in a handful of specialties. These specialties represent the core addressable market for the therapeutic area and should be prioritized in territory planning and sales force alignment.

## 5. Regional Analysis

Identifying which states generate the highest prescribing volume.

In [ ]:
top_states = (
    df.groupby("Prscrbr_State_Abrvtn")["Tot_Clms"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=top_states.values, y=top_states.index, color="darkorange", ax=ax)
ax.set_title("Top 10 States by Total Claims")
ax.set_xlabel("Total Claims")
ax.set_ylabel("State")
plt.tight_layout()
plt.show()

**Business interpretation:** Prescribing activity is unevenly distributed geographically, with a small number of states accounting for a large share of total claims. This regional concentration supports prioritizing sales resources and field team coverage in higher-opportunity states.

## 6. Correlation Analysis

Examining relationships between prescribing volume, cost, patient reach, and sales rep engagement (calls).

In [ ]:
corr_cols = ["Tot_Clms", "Tot_Drug_Cst", "Tot_Benes", "Calls_Made"]
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation Matrix: Prescribing Activity & Sales Engagement")
plt.tight_layout()
plt.show()

**Interpretation:** Total claims, drug cost, and beneficiary count are strongly positively correlated, confirming that prescribing volume, spend, and patient reach move together. The relationship between `Calls_Made` and prescribing metrics is comparatively weaker, suggesting sales call activity is not yet tightly aligned with physician value — a key finding for optimizing call planning and territory targeting.

## 7. Key Business Insights

- **Prescribing activity is concentrated** among a subset of physicians, who account for a disproportionate share of total claims and drug cost.
- **Higher prescription volume is associated with higher drug costs and larger patient panels**, confirming that claims, cost, and beneficiary counts move together.
- **Commercial opportunity varies meaningfully by specialty and region**, with a small number of specialties and states driving the majority of prescribing volume.
- **Sales call activity is only weakly correlated with prescribing value**, indicating potential misalignment between current rep effort and high-value physicians.
- **These findings motivate HCP segmentation** to systematically identify and prioritize high-value, high-opportunity physicians for targeted sales engagement — the focus of the subsequent SQL-based analysis.